# Stockfish 19 搜尋評估標註器 (`dataset_epd_wdl_20m.sf19.npz`) — Google Colab 專用版

本筆記本用於在 Google Colab 上執行 `tuner.data_pipeline.annotate_sf_labels`，將 `dataset_epd_wdl_20m.npz` 的局面交由 **Stockfish 19 (AVX2)** 進行固定節點搜尋，產出輕量級 sidecar 標註檔 `dataset_epd_wdl_20m.sf19.npz`（約 10～35 MB），並**自動定期同步回 Google Drive**：
`/content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz`

---

### 核心設計與斷線續傳機制
1. **智慧優先順序 (`build_priority_order`)**：
   - 第 1 順位的 **`1,000,000` 筆為 `dev` 驗證集**（與 `TexelTuningSession` 完全對齊）。
   - 第 2 順位的 **`10,000,000` 筆為 `train` 主力訓練子集**，接著才是剩餘 `train`。
   - 最後 `2,000,000` 筆 `unseen holdout` 預設跳過（`SKIP_HOLDOUT = True`，因為最終 Holdout Gate 永遠以純實戰 WDL 驗收）。
2. **隨時中斷、無縫續傳 (`resume=True`)**：
   - 每標註 `DRIVE_SYNC_EVERY = 200,000` 筆（約 3～4 分鐘）以及按下停止（`KeyboardInterrupt`）或完成時，皆會自動將最新 `.sf19.npz` 寫回 Google Drive。
   - 您可以**分階段累進標註**：例如先設 `LIMIT = 3_000_000`（標註 `1M dev + 2M train`）即可立刻在 `texel_tuner_runner.ipynb` 開啟 `USE_SF_LABELS = True` 調參；之後想擴充到 `LIMIT = 11_000_000`，只需改數字重跑，會直接從第 3,000,001 筆繼續！
3. **同時擷取 `sf_cp` 與 `sf_d1_cp`**：
   - 單次 UCI `go nodes` 搜尋同時記錄最終完成深度分數 (`sf_cp`) 與 `depth 1` 分數 (`sf_d1_cp`)，讓 `TexelTuningSession` 自動將戦術劇烈震盪局面（`|sf_cp - sf_d1_cp| > 180 cp`）的 $\lambda$ 降為 `0`，防止靜態 HCE 被迫擬合深層戰術陷阱。

In [1]:
# =====================================================================
# 檢視 Colab 雲端硬體規格與 CPU 核心數
# =====================================================================
!lscpu | grep "Model name\|Socket\|Thread\|NUMA\|CPU(s)"
!free -h
!python3 --version


CPU(s):                                  8
On-line CPU(s) list:                     0-7
Model name:                              AMD EPYC 7B12
Thread(s) per core:                      2
Socket(s):                               1
NUMA node(s):                            1
NUMA node0 CPU(s):                       0-7
               total        used        free      shared  buff/cache   available
Mem:            50Gi       1.4Gi        47Gi       2.2Mi       2.7Gi        49Gi
Swap:             0B          0B          0B
Python 3.13.18


In [6]:
# 00 — [Colab 專用] 檢查硬體規格、掛載 Google Drive、同步 Repo 與載入資料集
import os
import shutil
import subprocess
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    print("=== Colab CPU & RAM 規格 ===")
    subprocess.run(["bash", "-c", "lscpu | grep -E 'Model name|Socket|Thread|NUMA|CPU\\(s\\)'"], check=False)
    subprocess.run(["free", "-h"], check=False)

    drive.mount("/content/drive")
    DRIVE_HCE_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
    DRIVE_BIN_DIR = Path("/content/drive/MyDrive/chess_numba_data/bin")
    DRIVE_HCE_DIR.mkdir(parents=True, exist_ok=True)
    DRIVE_BIN_DIR.mkdir(parents=True, exist_ok=True)

    REPO_URL = "https://github.com/b08203031/chess_numba.git"
    BRANCH = "feat-initial-bitboard-setup"
    REPO_DIR = Path("/content/chess_numba")
    if not (REPO_DIR / ".git").is_dir():
        print(f"[Git] Cloning {REPO_URL} (branch={BRANCH}) -> {REPO_DIR} ...")
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        print(f"[Git] Syncing {REPO_DIR} -> origin/{BRANCH} ...")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-f", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
    os.chdir(REPO_DIR)

    LOCAL_HCE_DIR = REPO_DIR / "tuner" / "data" / "hce"
    LOCAL_HCE_DIR.mkdir(parents=True, exist_ok=True)

    for fname in (
        "dataset_epd_wdl_20m.npz",
        "dataset_epd_wdl_20m.sf19.npz",
        "dataset_epd_wdl_20m.sf19.ckpt_cp.npy",
        "dataset_epd_wdl_20m.sf19.ckpt_d1.npy",
    ):
        src = DRIVE_HCE_DIR / fname
        dst = LOCAL_HCE_DIR / fname
        if src.is_file():
            if not dst.is_file() or src.stat().st_size != dst.stat().st_size or ".sf19" in fname:
                print(f"📥 從 Google Drive 複製 {fname} ({src.stat().st_size / (1024**2):.1f} MB) -> 本機 SSD...")
                shutil.copy2(src, dst)
            else:
                print(f"✅ 本機 SSD 已存在 {fname} ({dst.stat().st_size / (1024**2):.1f} MB)")
        elif fname == "dataset_epd_wdl_20m.npz":
            raise FileNotFoundError(f"❌ 找不到 {src}！請確認已將 dataset_epd_wdl_20m.npz 放入 Google Drive 的 chess_numba_data/hce/ 資料夾。")
        elif fname == "dataset_epd_wdl_20m.sf19.npz":
            print(f"ℹ️ Google Drive 尚無既有 {fname}，將建立全新標註檔。")
else:
    print("ℹ️ 目前不在 Google Colab 環境，將使用本機路徑執行。")

=== Colab CPU & RAM 規格 ===
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Git] Syncing /content/chess_numba -> origin/feat-initial-bitboard-setup ...
📥 從 Google Drive 複製 dataset_epd_wdl_20m.npz (674.2 MB) -> 本機 SSD...
ℹ️ Google Drive 尚無既有 dataset_epd_wdl_20m.sf19.npz，將建立全新標註檔。


In [7]:
# 1 — 自動準備 Stockfish 19 (Linux x86-64 AVX2) 並快取至 Google Drive
import os
import platform
import shutil
import stat
import subprocess
import tarfile
import urllib.request
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "chess_engine").exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)

EXT_SF_DIR = REPO_ROOT / "external" / "stockfish"
EXT_SF_DIR.mkdir(parents=True, exist_ok=True)

if platform.system() == "Windows":
    SF_BIN_PATH = EXT_SF_DIR / "stockfish-windows-x86-64-avx2.exe"
else:
    SF_BIN_PATH = EXT_SF_DIR / "stockfish-ubuntu-x86-64-avx2"
    DRIVE_CACHED_SF = Path("/content/drive/MyDrive/chess_numba_data/bin/stockfish-ubuntu-x86-64-avx2")

    if not SF_BIN_PATH.is_file() and DRIVE_CACHED_SF.is_file():
        print(f"⚡ 從 Google Drive 快取載入 Stockfish 執行檔: {DRIVE_CACHED_SF}")
        shutil.copy2(DRIVE_CACHED_SF, SF_BIN_PATH)
        SF_BIN_PATH.chmod(SF_BIN_PATH.stat().st_mode | stat.S_IEXEC)

    if not SF_BIN_PATH.is_file():
        SF_URL = "https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar"
        tar_path = EXT_SF_DIR / "sf_avx2.tar"
        try:
            print(f"🌐 下載官方 Stockfish Linux AVX2 發行版: {SF_URL} ...")
            urllib.request.urlretrieve(SF_URL, tar_path)
            with tarfile.open(tar_path, "r") as tf:
                for member in tf.getmembers():
                    if member.isfile() and "stockfish" in Path(member.name).name:
                        extracted = tf.extractfile(member)
                        if extracted is not None:
                            with open(SF_BIN_PATH, "wb") as out_f:
                                shutil.copyfileobj(extracted, out_f)
                            break
            tar_path.unlink(missing_ok=True)
            SF_BIN_PATH.chmod(SF_BIN_PATH.stat().st_mode | stat.S_IEXEC)
        except Exception as dl_err:
            print(f"⚠️ 官方發行版下載失敗 ({dl_err})，改從官方原始碼編譯 AVX2 版本...")
            sf_src_dir = Path("/tmp/Stockfish_build")
            if not sf_src_dir.exists():
                subprocess.run(["git", "clone", "--depth", "1", "https://github.com/official-stockfish/Stockfish.git", str(sf_src_dir)], check=True)
            nproc = str(os.cpu_count() or 4)
            subprocess.run(["make", f"-j{nproc}", "build", "ARCH=x86-64-avx2"], cwd=str(sf_src_dir / "src"), check=True)
            shutil.copy2(sf_src_dir / "src" / "stockfish", SF_BIN_PATH)
            SF_BIN_PATH.chmod(SF_BIN_PATH.stat().st_mode | stat.S_IEXEC)

        if DRIVE_CACHED_SF.parent.exists() and SF_BIN_PATH.is_file():
            shutil.copy2(SF_BIN_PATH, DRIVE_CACHED_SF)
            print(f"💾 已將 Stockfish 執行檔快取至 Google Drive: {DRIVE_CACHED_SF}")

# 驗證 UCI 啟動與版本資訊
res = subprocess.run([str(SF_BIN_PATH)], input="uci\nquit\n", text=True, capture_output=True, timeout=10, check=True)
banner = res.stdout.splitlines()[0] if res.stdout else "Unknown"
print(f"✅ Stockfish 準備完成: {banner} ({SF_BIN_PATH})")

🌐 下載官方 Stockfish Linux AVX2 發行版: https://github.com/official-stockfish/Stockfish/releases/latest/download/stockfish-ubuntu-x86-64-avx2.tar ...
⚠️ 官方發行版下載失敗 (HTTP Error 404: Not Found)，改從官方原始碼編譯 AVX2 版本...
💾 已將 Stockfish 執行檔快取至 Google Drive: /content/drive/MyDrive/chess_numba_data/bin/stockfish-ubuntu-x86-64-avx2
✅ Stockfish 準備完成: Stockfish dev-20261110-def601cc by the Stockfish developers (see AUTHORS file) (/content/chess_numba/external/stockfish/stockfish-ubuntu-x86-64-avx2)


In [8]:
# 2 — 執行 Stockfish 19 搜尋標註 (支援隨時中斷 + 自動定期同步 Google Drive)
import inspect
import os
import shutil
import threading
from pathlib import Path
from tuner.data_pipeline.annotate_sf_labels import annotate_dataset

# ==================== 標註參數設定 ====================
# NODES: 每個局面的固定搜尋節點數
#   - 2000 ~ 2500: 推薦 Colab 高CP值設定 (SF19 NNUE 約達 depth 10~13，8核約 900~1,200 pos/s)
#   - 4000 ~ 5000: 更高深度設定 (8核約 450~650 pos/s)
NODES = 5000

# LIMIT: 依優先順序 (1M dev -> 10M train -> 其餘 train) 標註的總筆數上限
#   - 3_000_000  : 快速先導版 (完整標註 1M dev + 前 2M train，Colab 8核約 45~55 分鐘即可開始調參！)
#   - 6_000_000  : 進階版 (1M dev + 5M train，約 1.5~2 小時)
#   - 11_000_000 : 完整 Auto-RAM 訓練集版 (1M dev + 10M train，可分多次 Colab 續傳完成)
#   - 0          : 標註全部非 Holdout 局面 (17.64M 筆)
LIMIT = 0

SKIP_HOLDOUT = True          # Holdout Gate 永遠使用純 WDL 驗收，跳過 2M holdout 可省下時間
WORKERS = os.cpu_count() or 8  # Colab 上直接使用全部 vCPU 核心數
BATCH_SIZE = 1024
SAVE_EVERY = 100_000          # 本機 SSD memmap 刷新頻率
DRIVE_SYNC_EVERY = 200_000   # 每標註 20 萬筆 (~3 分鐘) 自動備份 .sf19.npz 到 Google Drive
# ======================================================

DATASET_PATH = Path("tuner/data/hce/dataset_epd_wdl_20m.npz")
LOCAL_OUT_PATH = Path("tuner/data/hce/dataset_epd_wdl_20m.sf19.npz")
DRIVE_HCE_DIR = Path("/content/drive/MyDrive/chess_numba_data/hce")
DRIVE_OUT_PATH = (
    DRIVE_HCE_DIR / "dataset_epd_wdl_20m.sf19.npz"
    if Path("/content/drive/MyDrive").exists()
    else None
)

def _backup_sf_files_to_drive() -> None:
    if DRIVE_OUT_PATH is None:
        return
    DRIVE_HCE_DIR.mkdir(parents=True, exist_ok=True)
    for f in (
        LOCAL_OUT_PATH,
        LOCAL_OUT_PATH.with_suffix(".ckpt_cp.npy"),
        LOCAL_OUT_PATH.with_suffix(".ckpt_d1.npy"),
    ):
        if f.is_file():
            try:
                tmp_dst = DRIVE_HCE_DIR / f"{f.name}.tmp"
                shutil.copy2(f, tmp_dst)
                tmp_dst.replace(DRIVE_HCE_DIR / f.name)
            except Exception:
                pass

_stop_sync = threading.Event()
def _periodic_drive_watcher() -> None:
    while not _stop_sync.wait(180.0):
        _backup_sf_files_to_drive()

_watcher = threading.Thread(target=_periodic_drive_watcher, daemon=True)
_watcher.start()

extra_kwargs = {}
sig_params = inspect.signature(annotate_dataset).parameters
if "drive_output_path" in sig_params:
    extra_kwargs["drive_output_path"] = DRIVE_OUT_PATH
    extra_kwargs["drive_sync_every"] = DRIVE_SYNC_EVERY

try:
    out_file = annotate_dataset(
        dataset_path=DATASET_PATH,
        output_path=LOCAL_OUT_PATH,
        stockfish_path=str(SF_BIN_PATH),
        nodes=NODES,
        workers=WORKERS,
        batch_size=BATCH_SIZE,
        save_every=SAVE_EVERY,
        limit=LIMIT,
        skip_holdout=SKIP_HOLDOUT,
        resume=True,
        hash_mb=64,
        **extra_kwargs,
    )
finally:
    _stop_sync.set()
    _backup_sf_files_to_drive()
    if DRIVE_OUT_PATH is not None and DRIVE_OUT_PATH.is_file():
        print(f"☁️ 已同步最新標註檔與 Checkpoint 至 Google Drive: {DRIVE_OUT_PATH}")

=== Stockfish Mixed-Label Sidecar Annotator ===
  Dataset   : tuner/data/hce/dataset_epd_wdl_20m.npz
  Output    : tuner/data/hce/dataset_epd_wdl_20m.sf19.npz
  Drive Sync: /content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz (every 200,000 rows)
  Engine    : Stockfish dev-20261110-def601cc by the Stockfish developers (see AUTHORS file) (/content/chess_numba/external/stockfish/stockfish-ubuntu-x86-64-avx2)
  Search    : 'go nodes 5000' | workers=8 | hash=64 MB/worker
  Loaded 19,627,749 bitboard rows in 7.5s
  Target pool: 17,627,749 rows (already done: 0, remaining todo: 17,627,749)
  [100.00%] +17,627,749/17,627,749 | 687 pos/s | ETA   0.0 min
Saved sidecar: tuner/data/hce/dataset_epd_wdl_20m.sf19.npz (55.93 MB) | annotated=17,627,749/19,627,749 (89.81%)
Synced to Drive: /content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz (55.93 MB)
  sf_cp stats : mean=+9.2 cp, std=2721.3 cp, p10=-480, p90=+492
  |sf - d1|   : median=38 cp, p90=200 cp, >180c

In [10]:
# 3 — 檢視 .sf19.npz 標註覆蓋率、自動校準 K_sf 預覽與選用下載至本機電腦
import numpy as np
from pathlib import Path
from tuner.tuner import split_dataset_indices, fit_best_k
from tuner.data_pipeline.annotate_sf_labels import UNANNOTATED_CP

DATASET_PATH = Path("tuner/data/hce/dataset_epd_wdl_20m.npz")
SIDECAR_PATH = Path("tuner/data/hce/dataset_epd_wdl_20m.sf19.npz")
DRIVE_SIDECAR = Path("/content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz")

with np.load(DATASET_PATH) as ds:
    wdl_results = ds["results"].astype(np.float64)
n_total = len(wdl_results)

with np.load(SIDECAR_PATH) as sf:
    sf_cp = sf["sf_cp"]
    sf_d1_cp = sf["sf_d1_cp"] if "sf_d1_cp" in sf.files else None
    nodes_meta = int(sf["nodes"]) if "nodes" in sf.files else -1
    banner_meta = str(sf["sf_banner"]) if "sf_banner" in sf.files else "Stockfish"

train_idx, dev_idx, holdout_idx = split_dataset_indices(n_total, 1_000_000, 2_000_000)
dev_valid = sf_cp[dev_idx] != UNANNOTATED_CP
train_valid = sf_cp[train_idx] != UNANNOTATED_CP
train_10m_valid = sf_cp[train_idx[:10_000_000]] != UNANNOTATED_CP

print(f"=== Sidecar 診斷報告 ({SIDECAR_PATH.name}) ===")
print(f"  標註引擎        : {banner_meta} (nodes={nodes_meta})")
print(f"  本機檔案大小    : {SIDECAR_PATH.stat().st_size / (1024**2):.2f} MB")
if DRIVE_SIDECAR.is_file():
    print(f"  雲端硬碟備份    : ✅ 已同步至 {DRIVE_SIDECAR} ({DRIVE_SIDECAR.stat().st_size / (1024**2):.2f} MB)")
print(f"  Dev 覆蓋率      : {int(np.sum(dev_valid)):,} / {len(dev_idx):,} ({100.0 * np.mean(dev_valid):.2f}%)")
print(f"  Train(前10M)覆蓋: {int(np.sum(train_10m_valid)):,} / 10,000,000 ({100.0 * np.mean(train_10m_valid):.2f}%)")
print(f"  Train(全部)覆蓋 : {int(np.sum(train_valid)):,} / {len(train_idx):,} ({100.0 * np.mean(train_valid):.2f}%)")

if np.any(dev_valid):
    sub_cp = sf_cp[dev_idx][dev_valid].astype(np.float64)
    sub_wdl = wdl_results[dev_idx][dev_valid]
    k_sf, mse_sf = fit_best_k(sub_cp[:200_000], sub_wdl[:200_000])
    print(f"  預估自動校準 K_sf: {k_sf:.4f} (SF19 對實戰 WDL MSE = {mse_sf:.6f})")

print("\n🚀 下一步：")
print("  1. 若要在 Colab 上調參：直接打開 texel_tuner_runner.ipynb，執行 # 00 會自動從 Google Drive 載入此 .sf19.npz，並在 # 2 設 USE_SF_LABELS = True！")
print("  2. 若要下載回本機電腦調參：將下方 DOWNLOAD_TO_LOCAL 改為 True 執行，或直接從 Google Drive (chess_numba_data/hce/) 下載 dataset_epd_wdl_20m.sf19.npz 放到本機 tuner/data/hce/。")

DOWNLOAD_TO_LOCAL = False
if DOWNLOAD_TO_LOCAL:
    try:
        from google.colab import files  # type: ignore
        files.download(str(SIDECAR_PATH))
    except ImportError:
        print("目前不在 Colab 環境，檔案已位於:", SIDECAR_PATH.resolve())

=== Sidecar 診斷報告 (dataset_epd_wdl_20m.sf19.npz) ===
  標註引擎        : Stockfish dev-20261110-def601cc by the Stockfish developers (see AUTHORS file) (nodes=5000)
  本機檔案大小    : 55.93 MB
  雲端硬碟備份    : ✅ 已同步至 /content/drive/MyDrive/chess_numba_data/hce/dataset_epd_wdl_20m.sf19.npz (55.93 MB)
  Dev 覆蓋率      : 1,000,000 / 1,000,000 (100.00%)
  Train(前10M)覆蓋: 10,000,000 / 10,000,000 (100.00%)
  Train(全部)覆蓋 : 16,627,749 / 16,627,749 (100.00%)
  預估自動校準 K_sf: 0.6546 (SF19 對實戰 WDL MSE = 0.099263)

🚀 下一步：
  1. 若要在 Colab 上調參：直接打開 texel_tuner_runner.ipynb，執行 # 00 會自動從 Google Drive 載入此 .sf19.npz，並在 # 2 設 USE_SF_LABELS = True！
  2. 若要下載回本機電腦調參：將下方 DOWNLOAD_TO_LOCAL 改為 True 執行，或直接從 Google Drive (chess_numba_data/hce/) 下載 dataset_epd_wdl_20m.sf19.npz 放到本機 tuner/data/hce/。


In [11]:
from google.colab import runtime

# 執行這行會直接中斷並釋放目前使用的虛擬機器運算資源
runtime.unassign()
